## Tools

Models can request to call tools that perform tasks such as fetching data from a database, searching the web or running the code. Tools are pairings of:
1. A schema, including the name of the tool, a description, and/or argument definitions(often a JSON schema)
2. A function or coroutine to execute

In [21]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
load_dotenv()

os.environ['GROQ_API_KEY'] = os.getenv('GROQ-API')
# print(os.environ['GROQ_API_KEY'])
model = ChatGroq(model = "qwen/qwen3.6-27b")
model = model.bind(max_tokens = 900)
# response = model.invoke("Why do parrots talk?")

# response

In [29]:
from langchain.tools import tool

@tool
def get_weather(location: str) -> str:
    """Get the weather at a location"""

    return f"It's sunny in {location}"

model_with_tools = model.bind_tools([get_weather])
model_with_tools

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.3', 'langchain': '1.4.0'}}, client=<groq.resources.chat.completions.Completions object at 0x74b46ad7c180>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x74b46ad7cb00>, model_name='qwen/qwen3.6-27b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'get_weather', 'description': 'Get the weather at a location', 'parameters': {'properties': {'location': {'type': 'string'}}, 'required': ['location'], 'type': 'object'}}}]}, config={}, config_factories=[])

In [30]:
response = model_with_tools.invoke("Whats the weather like in Boston?",max_tokens = 500)

print(response)

for tool_call in response.tool_calls:
    print(f'Tool: {tool_call['name']}')
    print(f'Args: {tool_call['args']}')

content='' additional_kwargs={'reasoning_content': 'Thinking Process:\n1.  Identify the user\'s request: "Whats the weather like in Boston?"\n2.  Identify the available tool: `get_weather` which takes a `location` parameter.\n3.  Extract the location: "Boston".\n4.  Call the tool: `get_weather(location="Boston")`.\n5.  Wait for the tool response (simulated/actual). I will generate the tool call now.✅\n', 'tool_calls': [{'id': 'n83jjy1b2', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 122, 'prompt_tokens': 275, 'total_tokens': 397, 'completion_time': 0.231454518, 'completion_tokens_details': {'reasoning_tokens': 94}, 'prompt_time': 0.022289263, 'prompt_tokens_details': None, 'queue_time': 0.051092317, 'total_time': 0.253743781}, 'model_name': 'qwen/qwen3.6-27b', 'system_fingerprint': 'fp_2f860a3fc2', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'mod

## Tool Execution Loops

In [32]:
# Step 1 Model Generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages, max_tokens = 500)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to the model for final response
final_response = model_with_tools.invoke(messages)
print(final_response)


content="It's sunny in Boston." additional_kwargs={'reasoning_content': 'The weather in Boston is sunny. I should report this to the user.\n'} response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 323, 'total_tokens': 348, 'completion_time': 0.049651711, 'completion_tokens_details': {'reasoning_tokens': 16}, 'prompt_time': 0.028097196, 'prompt_tokens_details': None, 'queue_time': 0.553386553, 'total_time': 0.077748907}, 'model_name': 'qwen/qwen3.6-27b', 'system_fingerprint': 'fp_60e0d2f248', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a09fcb-74c5-7113-a919-1a96f1da5b9c-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 323, 'output_tokens': 25, 'total_tokens': 348, 'output_token_details': {'reasoning': 16}}


In [33]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'Thinking Process:\n1.  Identify the user\'s request: "What\'s the weather in Boston?"\n2.  Identify the available tool: `get_weather` which takes a `location` parameter.\n3.  Extract the location: "Boston".\n4.  Call the tool: `get_weather(location="Boston")`.\n5.  Wait for the tool output and formulate the response. (Self-correction/Refinement: I will just generate the tool call now.) \nProceed. \nTool call: `get_weather` with `location: "Boston"`.✅\n', 'tool_calls': [{'id': 'jvf4kz48d', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 149, 'prompt_tokens': 275, 'total_tokens': 424, 'completion_time': 0.310362044, 'completion_tokens_details': {'reasoning_tokens': 121}, 'prompt_time': 0.024582766, 'prompt_tokens_details': None, 'queue_time': 0.052660743, 'to